# Investigation 4: Company Archetype Stability (Q4)

**Hypothesis:** Company behavioral archetypes (Aggressive Explorer / Selective Infiller / Dormant) derived from bid history are stable across consecutive sales — the same company falls in the same cluster regardless of which sale computes it.

**Decision Rule:**
- Agreement >= 80% → Stable. Use as predictive feature in model + UI badge.
- Agreement 60–80% → Moderate. Use as descriptive UI label only, not as model feature.
- Agreement < 60% → Unstable. Drop clustering. Expose raw behavioral metrics directly.

**Validation discipline:** Clustering is fit on Sales 257 and 261 independently. Dec 2025 bidders are assigned using Sale 261 centroids (`kmeans.predict()`), NOT re-clustered.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.optimize import linear_sum_assignment
import os
import re

# Paths
SHAPEFILE_PATH = '../../data/shapefiles/blocks.shp'
SALE_257_DIR = '../data/sale_257/'
SALE_261_DIR = '../data/sale_261/'
SALE_DEC2025_DIR = '../data/sale_obbba_dec2025/'

K = 3  # Number of clusters
RANDOM_STATE = 42

print('Configuration set.')

In [ ]:
# Load bid, company, and results data for both training sales
# TODO: Uncomment when data is downloaded
# Reuse parser functions from 01_adjacency_signal.ipynb
#
# df_bid_257 = load_bid_file(SALE_257_DIR)
# df_com_257 = load_com_file(SALE_257_DIR)
# df_res_257 = load_res_file(SALE_257_DIR)
# df_trt_257 = load_trt_file(SALE_257_DIR)
#
# df_bid_261 = load_bid_file(SALE_261_DIR)
# df_com_261 = load_com_file(SALE_261_DIR)
# df_res_261 = load_res_file(SALE_261_DIR)
# df_trt_261 = load_trt_file(SALE_261_DIR)

print('Data loading cells ready.')

## 2. Compute Behavioral Features (Sale 257)

For each company active in Sale 257, compute four behavioral features:
1. `bid_frequency` = number of bids / number of available blocks
2. `win_rate` = accepted high bids / total bids placed
3. `avg_bid_premium` = mean(bid_amount) / mean(winning_bid for same blocks)
4. `water_depth_preference_mean` = mean water depth of blocks bid on

In [ ]:
def compute_behavioral_features(df_bid, df_res, df_trt):
    """
    Compute behavioral features per company from a single sale's data.
    
    Returns DataFrame with columns:
        company_id, bid_frequency, win_rate, avg_bid_premium, water_depth_pref
    """
    # TODO: Implement feature computation
    # Steps:
    # 1. Count total available blocks from TRT file
    # 2. For each company:
    #    a. bid_frequency = len(bids by company) / total_available_blocks
    #    b. Identify which bids were high bids from RES file
    #       win_rate = high_bids_accepted / total_bids
    #    c. For each block bid on, get the winning bid amount from RES
    #       avg_bid_premium = mean(bid_amount / winning_bid_for_same_block)
    #    d. Join bid blocks to TRT for water depth
    #       water_depth_pref = mean(water_depth of blocks bid on)
    # 3. Return feature DataFrame
    pass

# features_257 = compute_behavioral_features(df_bid_257, df_res_257, df_trt_257)
# print(f'Computed features for {len(features_257)} companies from Sale 257')
# features_257.describe()

## 3. K-Means Clustering (Sale 257)

Cluster companies into 3 archetypes based on behavioral features.
Label clusters based on centroid characteristics.

In [ ]:
# TODO: Cluster Sale 257 companies
# Steps:
# 1. Scale features: scaler_257 = StandardScaler().fit_transform(features_257[feature_cols])
# 2. Fit K-means: kmeans_257 = KMeans(n_clusters=K, random_state=RANDOM_STATE).fit(scaled_257)
# 3. Assign labels: features_257['cluster_257'] = kmeans_257.labels_
# 4. Inspect centroids to assign meaningful labels:
#    - Highest bid_frequency + highest avg_bid_premium → "Aggressive Explorer"
#    - Moderate bid_frequency + high win_rate → "Selective Infiller"
#    - Low bid_frequency → "Dormant"
#
# Scatter plot: bid_frequency vs. win_rate, colored by cluster
# fig, ax = plt.subplots(figsize=(10, 7))
# scatter = ax.scatter(features_257['bid_frequency'], features_257['win_rate'],
#                      c=features_257['cluster_257'], cmap='Set1', alpha=0.7)
# ax.set_xlabel('Bid Frequency')
# ax.set_ylabel('Win Rate')
# ax.set_title('Company Archetypes — Sale 257')
# plt.colorbar(scatter, label='Cluster')
pass

## 4. Repeat for Sale 261

Same feature computation and independent K-means clustering on Sale 261 data.

In [ ]:
# TODO: Repeat Steps 2 & 3 for Sale 261
# features_261 = compute_behavioral_features(df_bid_261, df_res_261, df_trt_261)
# scaler_261 = StandardScaler()
# scaled_261 = scaler_261.fit_transform(features_261[feature_cols])
# kmeans_261 = KMeans(n_clusters=K, random_state=RANDOM_STATE).fit(scaled_261)
# features_261['cluster_261'] = kmeans_261.labels_
#
# Scatter plot for Sale 261
pass

## 5. Agreement Rate

Compute what percentage of companies active in BOTH sales receive the same archetype label.
Use the Hungarian algorithm to resolve label permutation between the two independent clusterings.

In [ ]:
# TODO: Compute agreement rate
# Steps:
# 1. Identify companies present in both sales
#    shared = set(features_257.company_id) & set(features_261.company_id)
#
# 2. Build confusion matrix (3x3) of (cluster_257, cluster_261) for shared companies
#    from sklearn.metrics import confusion_matrix
#    cm = confusion_matrix(labels_257_shared, labels_261_shared, labels=[0,1,2])
#
# 3. Hungarian algorithm to find optimal label mapping
#    row_ind, col_ind = linear_sum_assignment(-cm)  # negative because we want to maximize
#    label_map = dict(zip(col_ind, row_ind))
#
# 4. Remap Sale 261 labels and compute agreement
#    remapped_261 = [label_map[l] for l in labels_261_shared]
#    agreement = sum(l257 == l261_r for l257, l261_r in zip(labels_257_shared, remapped_261)) / len(shared)
#
# print(f'Shared companies: {len(shared)}')
# print(f'Agreement rate: {agreement:.1%}')
pass

In [ ]:
# Confusion matrix heatmap
# fig, ax = plt.subplots(figsize=(8, 6))
# im = ax.imshow(cm, cmap='Blues')
# ... annotate cells ...
# ax.set_xlabel('Sale 261 Cluster')
# ax.set_ylabel('Sale 257 Cluster')
# ax.set_title(f'Archetype Agreement: {agreement:.1%}')
# plt.savefig('../outputs/q4_archetype_confusion.png', dpi=150, bbox_inches='tight')
pass

## 6. Switcher Analysis

For companies that switch archetypes between Sales 257 and 261:
- List them with their old and new labels
- Investigate whether switches are explained by known events (M&A, exploration results, commodity prices)

In [ ]:
# TODO: Identify and analyze switchers
# switchers = shared_df[shared_df['cluster_257_remapped'] != shared_df['cluster_261']]
# print(f'{len(switchers)} companies switched archetypes:')
# display(switchers[['company_name', 'cluster_257_label', 'cluster_261_label']])
pass

## 7. Geographic Signature

For each archetype, plot the geographic distribution of bids.
- Do Aggressive Explorers cluster in specific water depths or planning areas?
- Do Selective Infillers systematically target adjacency opportunities?

In [ ]:
# TODO: Geographic signature analysis
# For each archetype:
#   1. Get all blocks bid on by companies in this archetype
#   2. Plot on GOM map
#   3. Compute summary stats: mean water depth, planning area distribution
#
# fig, axes = plt.subplots(1, 3, figsize=(24, 8))
# for i, archetype in enumerate(['Aggressive Explorer', 'Selective Infiller', 'Dormant']):
#     ... plot bids by companies in this archetype on GOM map ...
pass

---
## 8. VALIDATION — HELD-OUT (December 2025 Sale)

**⚠️ Assign Dec 2025 bidders to archetypes using the Sale 261 model (kmeans_261.predict()), NOT by re-clustering.**

In [ ]:
# TODO: Validate on Dec 2025
# 1. Compute behavioral features for Dec 2025 bidders
# 2. Scale using Sale 261 scaler: scaled_dec2025 = scaler_261.transform(features_dec2025[feature_cols])
# 3. Predict clusters: features_dec2025['cluster'] = kmeans_261.predict(scaled_dec2025)
# 4. Map predicted clusters onto GOM grid
#
# Note: This tests whether the Sale 261 clustering model generalizes to a new sale
# under a different regulatory regime (OBBBA vs IRA).
pass

## 9. Findings

### Results
- **Shared companies across Sales 257 and 261:** [TBD]
- **Agreement rate (after Hungarian relabeling):** [TBD]%
- **Cluster centroids (Sale 257):** [TBD table]
- **Cluster centroids (Sale 261):** [TBD table]
- **Geographic signature present:** [TBD yes/no per archetype]

### Interpretation
[TBD — fill in based on results and PRD decision rule]

### Recommendation
[TBD — Use as predictive feature / Descriptive label only / Drop clustering]